# Task 1.1 — MSVMpack: Step-by-Step Method Description

## Step 1 — Input Space & Kernel → RKHS H_κ &nbsp;*(Section 2)*
You begin with a labelled training set $\{(x_i,y_i)\}_{i=1}^n$ where $x_i\in\mathcal{X}\subseteq\mathbb{R}^d$ and $y_i\in\{1,\dots,Q\}$.
A positive-definite kernel $\kappa:\mathcal{X}\times\mathcal{X}\to\mathbb{R}$ (e.g. RBF, polynomial) is chosen, implicitly defining a Reproducing Kernel Hilbert Space (RKHS) $\mathcal{H}_\kappa$.
All computations happen via dot-products $\langle\Phi(x_i),\Phi(x_j)\rangle_{\mathcal{H}_\kappa}=\kappa(x_i,x_j)$ — the kernel trick.

## Step 2 — Function Class & Decision Rule &nbsp;*(Section 2)*
The classifier is a vector-valued function $h=(h_1,\dots,h_Q)\in\mathcal{H}_\kappa^Q$.
Each component $h_k(x)=\sum_i c_{ik}\kappa(x_i,x)+b_k$ scores class $k$ for input $x$.
The prediction is $\hat{y}=\arg\max_{k=1}^{Q}\,h_k(x)$ — the class with the highest score wins.

## Step 3 — Generic QP Formulation &nbsp;*(Definition 1, Eq. 1)*
MSVMpack frames all four M-SVMs as one **generic Quadratic Programme**:
$$\min_{h,\xi}\;\frac{\lambda}{2}\overline{\|h\|}_M^2+\frac{1}{n}\sum_{i=1}^n\sum_{k\ne y_i}\xi_{ik}$$
subject to $\;K_1\,h_{y_i}(x_i)-h_k(x_i)\ge K_2-\xi_{ik}$, $\;\xi_{ik}\ge 0$, $\forall i,k\ne y_i$.

$\lambda$ regularises complexity; $M$ defines the norm on $h$; $K_1,K_2$ set margin geometry; $\xi_{ik}$ are slack variables allowing soft margins.

## Step 4 — Specialisation to 4 M-SVMs via $(M,p,K_1,K_2,K_3)$ &nbsp;*(Table 1)*
| Model | $K_1$ | $K_2$ | Norm matrix $M$ | Loss $p$ |
|-------|--------|--------|----------------|----------|
| Weston & Watkins (WW) | 1 | 2 | $I-\tfrac{1}{Q}\mathbf{1}\mathbf{1}^\top$ | 1 |
| Crammer & Singer (CS) | 1 | 1 | $I$ | 1 |
| Lee-Lin-Wahba (LLW) | 1 | 2 | $I$ | 2 |
| MSVM2 (Guermeur) | $\tfrac{Q-1}{Q}$ | $\tfrac{Q-1}{Q}$ | custom | 2 |

Plugging the right constants into Definition 1 gives the exact convex QP for each model — all sharing one solver.

## Step 5 — Wolfe Dual & Lagrange Multipliers $\alpha_{ik}$ &nbsp;*(Section 3.1)*
Forming the Lagrangian and applying KKT optimality conditions yields the **Wolfe dual**: a maximisation over dual variables $\alpha_{ik}\ge 0$ (one per training example–class pair $k\ne y_i$).
The primal weight vectors disappear and the kernel matrix $K_{ij}=\kappa(x_i,x_j)$ appears in their place, so we never need to work in the high-dimensional feature space explicitly.

## Step 6 — Frank-Wolfe Descent + LP Sub-problem &nbsp;*(Section 3.1)*
Directly solving the dual QP is $O(n^2)$ in memory. MSVMpack uses the **Frank-Wolfe (conditional gradient)** method instead:
at iteration $t$, linearise the dual objective at $\alpha^{(t)}$ and solve a cheap **Linear Programme (LP)** to get descent direction $d^{(t)}$.
A line-search gives step size $\eta^{(t)}$, then $\alpha^{(t+1)}=\alpha^{(t)}+\eta^{(t)}d^{(t)}$.
Each LP is far cheaper than the full QP and has closed-form structure due to the polytope constraints.

## Step 7 — Decomposition for Large Datasets &nbsp;*(Section 3.1)*
For very large $n$ (e.g. CB513: 84 K samples), even one LP is expensive. MSVMpack applies a **working-set decomposition**: each Frank-Wolfe iteration updates only a small active subset of examples while the rest are frozen.
Memory stays proportional to the working-set size, not $n^2$. The algorithm cycles through subsets, guaranteeing eventual global convergence.

## Step 8 — Stopping Criterion via Upper Bound $U(\alpha)$ &nbsp;*(Section 3.1)*
Training stops when the **duality gap** $U(\alpha)-D(\alpha)<\varepsilon$, where $D(\alpha)$ is the current dual value and $U(\alpha)$ is an analytically computable upper bound on the primal optimum.
This gives a principled, model-specific criterion — no arbitrary iteration limit needed. For MSVM2 with $Q>20$, computing $U(\alpha)$ requires an embedded QP and can become the dominant cost.

## Step 9 — Prediction &nbsp;*(Section 2)*
Store only the non-zero $\alpha_{ik}$ (support vectors). For test point $x^*$:
$$h_k(x^*)=\sum_{i:\alpha_{ik}>0}c_{ik}\,\kappa(x_i,x^*)+b_k,\qquad\hat{y}=\arg\max_k h_k(x^*).$$

## Summary
MSVMpack solves the problem that prior M-SVM implementations were fragmented and memory-limited by providing a **single unified C package** with a decomposition Frank-Wolfe solver covering all four classical formulations — making large-scale multi-class SVM tractable where isolated tools like BSVM fail on memory.